# Data Cleaning

## Objective

The objective of this notebook is to clean, transform, and prepare the Olist e-commerce datasets for further analysis.

The data wrangling process includes:

- Data quality assessment
- Missing values treatment
- Duplicate removal
- Data type correction
- Feature engineering
- Dataset integration
- Preparation of analytical datasets for SQL and dashboard development

The final cleaned datasets will be used for business analysis and visualization.

# 1. Import Libraries

The following libraries are used for data manipulation, cleaning, and validation.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File management
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)

# 2. Load Raw Datasets

The Olist dataset contains multiple relational tables representing different parts of the e-commerce business.

The datasets include:

- Customers
- Orders
- Order Items
- Payments
- Products
- Reviews
- Sellers

Each dataset will be analyzed and cleaned before integration.

In [2]:
customers = pd.read_csv("../data/RAW/olist_customers_dataset.csv")
orders = pd.read_csv("../data/RAW/olist_orders_dataset.csv")
items = pd.read_csv("../data/RAW/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/RAW/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/RAW/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/RAW/olist_products_dataset.csv")
sellers = pd.read_csv("../data/RAW/olist_sellers_dataset.csv")
category = pd.read_csv("../data/RAW/product_category_name_translation.csv")
geolocation = pd.read_csv("../data/RAW/olist_geolocation_dataset.csv")

# Dataset Overview

In [3]:
datasets = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "category": category,
    "geolocation": geolocation
}


for name, df in datasets.items():
    
    print(
        f"{name}: {df.shape[0]} rows and {df.shape[1]} columns"
    )

customers: 99441 rows and 5 columns
orders: 99441 rows and 8 columns
items: 112650 rows and 7 columns
payments: 103886 rows and 5 columns
reviews: 99224 rows and 7 columns
products: 32951 rows and 9 columns
sellers: 3095 rows and 4 columns
category: 71 rows and 2 columns
geolocation: 1000163 rows and 5 columns


# 3. Create Cleaning Copies

The raw datasets are preserved by creating independent copies.

All cleaning and transformation steps will be performed on these copies to maintain the original data.

In [4]:
customers_clean = customers.copy()

orders_clean = orders.copy()

items_clean = items.copy()

payments_clean = payments.copy()

reviews_clean = reviews.copy()

products_clean = products.copy()

sellers_clean = sellers.copy()

category_clean = category.copy()

geolocation_clean = geolocation.copy()

# 4. Data Cleaning

## 4.1 Customers Data Cleaning

The customers dataset contains information about customers registered on the marketplace.

The cleaning process focuses on:

- Standardizing text columns
- Correcting data formats
- Preparing location information for customer analysis

No missing values or duplicated records require treatment based on the previous exploratory analysis.

### 4.1.1 Standardize city

In [5]:
# Standardize customer city names

customers_clean["customer_city"] = (
    customers_clean["customer_city"]
    .str.lower()
    .str.strip()
)

City names are standardized to avoid duplicated categories caused by different text formats.

### 4.1.2 Standardize state

In [6]:
# Standardize state abbreviations

customers_clean["customer_state"] = (
    customers_clean["customer_state"]
    .str.upper()
    .str.strip()
)

State abbreviations are converted to uppercase to maintain consistency during regional analysis.

### 4.1.3 ZIP code type conversion

In [7]:
# Convert ZIP code prefix to string

customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"]
    .astype(str)
)

The ZIP code prefix is converted to string because it represents a geographic identifier rather than a numerical measurement.

In [8]:
# Validation of the transformations

customers_clean.dtypes

customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
dtype: object

## 4.2 Orders Data Cleaning

The orders dataset contains information about the complete order lifecycle, including purchase, approval, shipping, and delivery timestamps.

The cleaning process includes:

- Converting date columns to datetime format
- Investigating missing timestamps
- Preserving valid business scenarios
- Preparing delivery metrics

### 4.2.1 Convert Timestamp Columns

In [9]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean[date_columns] = (
    orders_clean[date_columns]
    .apply(pd.to_datetime)
)

Timestamp columns are converted to datetime format to enable calculations involving delivery time and order trends.

### 4.2.2 Investigate Missing order_approved_at

In [10]:
orders_clean[
    orders_clean["order_approved_at"].isna()
]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Missing approval timestamps were investigated according to order status.

Most missing values are related to canceled or incomplete orders.

A small number of delivered orders contain missing approval timestamps, indicating possible data quality inconsistencies. These records will be preserved and only excluded from analyses requiring complete approval information.

### 4.2.3 Investigate Missing order_delivered_carrier_date

In [11]:
orders_clean[
    orders_clean["order_delivered_carrier_date"].isna()
]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Missing carrier delivery dates are mainly associated with orders that were canceled, unavailable, or still being processed.

These missing values are expected because these orders did not reach the shipping stage.

Only two delivered orders contain missing carrier dates, which represent minor inconsistencies in the dataset.

### 4.2.4 Investigate Missing order_delivered_customer_date

In [12]:
orders_clean[
    orders_clean["order_delivered_customer_date"].isna()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

### 4.2.5 Missing Values Treatment Decision

Missing customer delivery dates are mostly related to orders that were not completed or are still in progress.

Orders with status "shipped" do not have a delivery date because the delivery process has not been completed yet.

The eight delivered orders without delivery timestamps were considered data quality inconsistencies and will be excluded only from delivery performance calculations.

## 4.3 Order Items Data Cleaning

The order items dataset contains information about products included in each order, including product identifiers, seller information, prices, and freight values.

Based on the exploratory analysis, the dataset does not contain missing values or duplicated records.

The cleaning process includes:

- Converting date columns to datetime format
- Validating numerical fields
- Preparing the dataset for sales and product performance analysis

### 4.3.1 Convert Timestamp Columns

The shipping limit date column is converted to datetime format to enable time-based analysis and support future logistics performance calculations.

In [13]:
items_clean["shipping_limit_date"] = pd.to_datetime(
    items_clean["shipping_limit_date"]
)

### 4.3.2 Validate Numerical Fields

Numerical columns are validated to ensure that monetary values are consistent and suitable for sales analysis.

The price and freight value columns are checked for invalid negative values.

In [14]:
items_clean[
    [
        "price",
        "freight_value"
    ]
].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [15]:
items_clean[
    (items_clean["price"] < 0) |
    (items_clean["freight_value"] < 0)
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


There are no invalid negative values in both price and freight columns.

### 4.3.3 Validate Identifier Columns

Identifier columns are validated because they will be used to connect the order items dataset with orders, products, and sellers tables during data integration.

In [16]:
items_clean[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id"
    ]
].isnull().sum()

order_id         0
order_item_id    0
product_id       0
seller_id        0
dtype: int64

### 4.3.3 Final Validation

After applying the transformations, the dataset structure is reviewed to confirm that the cleaning steps were successfully completed.

In [17]:
items_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [18]:
items_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


# 4.4 Payments Data Cleaning

The payments dataset was cleaned based on the issues identified during the data exploration phase.

The following validations and cleaning steps were performed:
- Payment type validation
- Payment value validation
- Payment installments validation
- Investigation of inconsistent payment records
- Removal of invalid records
- Final validation after cleaning

## 4.4.1 Payment Type Validation

The payment type column was analyzed to verify whether all categories represent valid payment methods.

An undefined category was identified and investigated before deciding the appropriate treatment.

In [19]:
payments_clean["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

### 4.4.2 Payment Value Validation

The payment value column was analyzed to verify whether all values represent valid transactions.

Negative payment values were checked because they would indicate inconsistencies in the dataset.

In [20]:
payments_clean[
    payments_clean["payment_value"] < 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


### 4.4.3 Payment Installments Validation


The payment installments column was analyzed to verify whether all values represent valid installment quantities.

Negative values and zero installments were investigated because installment quantities should be positive values.

In [21]:
payments_clean["payment_installments"].value_counts().sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

In [22]:
# Check negative installments
payments_clean[
    payments_clean["payment_installments"] < 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


The payment installments column was analyzed to verify whether all values represent valid installment quantities.

The distribution of installment values was inspected, and two records with `payment_installments = 0` were identified.

These records were further investigated by checking their payment information and order status.

In [23]:
payments_clean[
    payments_clean["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [24]:
orders_clean[
    orders_clean["order_id"].isin(
        payments_clean[
            payments_clean["payment_installments"] == 0
        ]["order_id"]
    )
][
    ["order_id", "order_status"]
]

,order_id,order_status
63782,744bade1fcf9ff3f31d860ace076d422,delivered
66368,1a57108394169c0b47d8f876acc9ba2d,delivered


In [25]:
payments_clean[
    payments_clean["order_id"].isin(
        payments_clean[
            payments_clean["payment_installments"] == 0
        ]["order_id"]
    )
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


Treatment

The investigation confirmed that these records correspond to completed credit card transactions.

Since a credit card payment must have at least one installment, the inconsistent values were corrected from 0 to 1 instead of removing the records.

The identified records were compared with the orders dataset to verify whether they represented valid transactions.

In [26]:
payments_clean.loc[
    payments_clean["payment_installments"] == 0,
    "payment_installments"
] = 1

In [27]:
payments_clean[
    payments_clean["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


### 4.4.4 Investigation of Inconsistent Payment Records

The records with `payment_type = 'not_defined'` were identified and analyzed to understand their context.

The related orders were matched with the orders dataset using the `order_id` column, and the order status was inspected.

In [28]:
payments_clean[
    payments_clean["payment_type"] == "not_defined"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


The orders associated with undefined payment types were identified by comparing `order_id` values between the payments and orders datasets.

The order status was analyzed to better understand the nature of these records.

In [29]:
orders_clean[
    orders_clean["order_id"].isin(
        payments_clean[
            payments_clean["payment_type"] == "not_defined"
        ]["order_id"]
    )
][
    ["order_id", "order_status"]
]

,order_id,order_status
1130,00b1cb0320190ca0daa2c88b35206009,canceled
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled


Removing Invalid Payment Records

The analysis identified the order status of records associated with `not_defined` payment types.

All identified orders were classified as `canceled`, providing additional context about these payment records.


In [30]:
payments_clean = payments_clean[
    payments_clean["payment_type"] != "not_defined"
]

### 4.4.5 Standardize Payment Type

The payment type column is standardized to ensure consistency between categories and avoid duplicated categories caused by different text formats.

In [31]:
payments_clean["payment_type"] = (
    payments_clean["payment_type"]
    .str.lower()
    .str.strip()
)

## 4.5 Review Data Cleaning

The review dataset contains information about customer feedback, including review scores, written comments, and review timestamps.

The main objectives of this cleaning step are to:

- Convert date columns to the appropriate datetime format.
- Validate customer review scores.
- Investigate missing values.
- Validate review and order identifiers.
- Check for duplicate records.
- Preserve valid missing values where they represent legitimate customer behavior.

Maintaining the original meaning of the data is an important part of the cleaning process. Therefore, values will only be modified or removed when there is sufficient evidence that they represent a data quality issue.

### 4.5.1 Convert Date Columns

The dataset contains two columns related to review dates:

- `review_creation_date`: the date when the review was created.
- `review_answer_timestamp`: the timestamp associated with the review response.

These columns are initially stored as text values. Converting them to the datetime data type makes the data more suitable for time-based analysis.

This conversion will allow us to calculate time differences and analyze review activity by day, month, or year in later stages of the project.

In [32]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_columns:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col],
        errors="coerce"
    )

reviews_clean[review_date_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Both review date columns were successfully converted to the datetime data type:

- `review_creation_date` → `datetime64[us]`
- `review_answer_timestamp` → `datetime64[us]`

The conversion was successful, allowing these columns to be used for time-based analysis, such as analyzing review trends over time or calculating the time between review creation and response.

No additional treatment was required.

### 4.5.2 Investigate Missing Values

### 4.5.2.1 Identify Missing Values

Missing values are common in real-world datasets and should not automatically be considered data quality issues.

The first step is to identify which columns contain missing values and measure their frequency. This provides an overview of where missing data occurs before any treatment is applied.

For the review dataset, missing values are particularly relevant because written customer feedback is optional. Therefore, each affected column will be investigated individually before deciding whether the missing values should be retained, imputed, or removed.

In [33]:
review_missing_summary = pd.DataFrame({
    "missing_count": reviews_clean.isna().sum(),
    "missing_percentage": (
        reviews_clean.isna().mean() * 100
    ).round(2)
})

review_missing_summary = review_missing_summary[
    review_missing_summary["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

review_missing_summary

,missing_count,missing_percentage
review_comment_title,87656,88.34
review_comment_message,58247,58.70


Missing values were found only in the two customer-written feedback fields:

- `review_comment_title`: 87,656 missing values (88.34%)
- `review_comment_message`: 58,247 missing values (58.70%)

Both fields contain optional written feedback, so the high percentage of missing values requires further investigation before determining whether any treatment is necessary.

### 4.5.2.2 Investigate Reviews Without a Comment Message

The `review_comment_message` field contains optional written feedback from customers.

A missing message does not necessarily mean that the review is incomplete, because customers can provide a numerical score without writing a comment.

To investigate this behavior, we will compare the review scores of records with and without a `review_comment_message`.

In [34]:
reviews_clean.groupby(
    reviews_clean["review_comment_message"].isna()
)["review_score"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
review_comment_message,,,,
False,40977,3.669864,1,5
True,58247,4.379470,1,5


The analysis shows that reviews with missing `review_comment_message` still contain valid review scores ranging from 1 to 5.

Reviews with a written comment have an average score of 3.67, while reviews without a written comment have a higher average score of 4.38.

This confirms that missing review comments do not indicate invalid review records. The difference in average scores is also potentially relevant for the exploratory analysis phase, where we can investigate whether customers who provide written feedback tend to have different satisfaction levels.

For the data cleaning stage, the missing comment values will therefore be retained.

### 4.5.2.3 Identify Reviews With No Written Comment

The review dataset contains two separate fields for written feedback:

- `review_comment_title`
- `review_comment_message`

A customer may provide a title, a message, both, or neither.

Therefore, we will identify reviews where both fields are missing to determine how many customers provided only a numerical rating without any written feedback.

In [35]:
comment_columns = [
    "review_comment_title",
    "review_comment_message"
]

no_comment = reviews_clean[comment_columns].isna().all(axis=1)

print("Reviews with no written comment:", no_comment.sum())
print(
    "Percentage:",
    round(no_comment.mean() * 100, 2),
    "%"
)

Reviews with no written comment: 56518
Percentage: 56.96 %


A total of 56,518 reviews, representing 56.96% of the dataset, contain no written feedback in either `review_comment_title` or `review_comment_message`.

This indicates that providing written feedback is not a necessary part of submitting a review. Customers can provide a numerical score without adding any written comments.

### 4.5.2.4 Analyze Comment Combinations

To better understand the structure of the missing values, reviews will be classified into four categories based on the presence of the title and message fields:

- **Title and message:** both fields are present.
- **Title only:** only the title is present.
- **Message only:** only the message is present.
- **No comment:** both fields are missing.

This classification allows us to determine whether the missing values follow expected patterns in customer behavior.

In [36]:
reviews_clean["comment_status"] = np.select(
    [
        reviews_clean["review_comment_title"].notna()
        & reviews_clean["review_comment_message"].notna(),

        reviews_clean["review_comment_title"].notna()
        & reviews_clean["review_comment_message"].isna(),

        reviews_clean["review_comment_title"].isna()
        & reviews_clean["review_comment_message"].notna(),

        reviews_clean["review_comment_title"].isna()
        & reviews_clean["review_comment_message"].isna()
    ],
    [
        "Title and message",
        "Title only",
        "Message only",
        "No comment"
    ],
    default="Unknown"
)

reviews_clean["comment_status"].value_counts()

comment_status
No comment           56518
Message only         31138
Title and message     9839
Title only            1729
Name: count, dtype: int64

In [37]:
reviews_clean["comment_status"].value_counts(
    normalize=True
).mul(100).round(2)

comment_status
No comment           56.96
Message only         31.38
Title and message     9.92
Title only            1.74
Name: proportion, dtype: float64

The analysis shows that:

- 56.96% of reviews contain no written comments.
- 31.38% contain only a message.
- 9.92% contain both a title and a message.
- 1.74% contain only a title.

The results show that written feedback is optional and that the absence of a title or message follows a clear pattern rather than appearing to be caused by random data loss.

### 4.5.2.5 Compare Review Scores by Comment Status

After classifying reviews according to their written feedback, we will compare review scores across the different comment categories.

This analysis helps determine whether customers who provide written feedback have different rating patterns from customers who only provide a numerical score.

The analysis is exploratory and does not imply that the presence or absence of a comment causes a particular review score.

In [38]:
reviews_clean.groupby("comment_status")["review_score"].agg(
    ["count", "mean", "median", "min", "max"]
).sort_values(
    "count",
    ascending=False
)

,count,mean,median,min,max
comment_status,,,,,
No comment,56518,4.375916,5.0,1,5
Message only,31138,3.616867,4.0,1,5
Title and message,9839,3.837585,5.0,1,5
Title only,1729,4.495662,5.0,1,5


The review score distribution differs across comment categories.

Reviews without written comments have an average score of 4.38, while reviews containing only a message have the lowest average score at 3.62.

Reviews containing only a title have the highest average score at 4.50. However, this group represents only 1.74% of all reviews, so conclusions based on this group should be treated with caution.

These differences may provide useful business insights during the exploratory analysis stage. However, they do not indicate a data quality problem and therefore do not justify changing the missing values.

### 4.5.2.6 Missing Values Treatment Decision

The investigation indicates that missing values in `review_comment_title` and `review_comment_message` represent valid customer behavior rather than data quality issues.

Customers can submit a numerical review score without providing written feedback. More than half of the reviews (56.96%) contain neither a title nor a message, which further supports the conclusion that written comments are optional.

Therefore, the following treatment decisions were made:

- Missing values in `review_comment_title` will be retained.
- Missing values in `review_comment_message` will be retained.
- No artificial values will be used to replace missing comments.
- No rows will be removed because of missing comment fields.

The relationship between written feedback and review scores will be revisited during the Exploratory Data Analysis stage.

In [39]:
reviews_clean.drop(
    columns=["comment_status"],
    inplace=True
)

In [40]:
"comment_status" in reviews_clean.columns

False

### 4.5.3 Validate Review Scores

The `review_score` column represents the numerical rating provided by the customer.

Valid review scores should range from 1 to 5. Values outside this range would indicate invalid records and could affect customer satisfaction metrics.

We will first examine the distribution of the review scores and then check for values outside the expected range.

In [41]:
# Examine the distribution of review scores

reviews_clean["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [42]:
# Check for invalid review scores

invalid_scores = reviews_clean[
    ~reviews_clean["review_score"].between(1, 5)
]

print(
    f"Number of invalid review scores: {len(invalid_scores)}"
)

Number of invalid review scores: 0


No invalid review scores were identified.

All 99,224 review records contain a score between 1 and 5, which is consistent with the expected rating scale. Therefore, no values were modified or removed from the `review_score` column.

The strong concentration of reviews with a score of 5 will be considered during the Exploratory Data Analysis stage, where customer satisfaction patterns will be investigated in greater detail.

### 4.5.4 Validate Identifier Columns

### 4.5.4.1 Check Missing Identifiers

The review dataset contains two important identifier columns:

- `review_id`: identifies the review record.
- `order_id`: links the review to the corresponding customer order.

These identifiers are essential for connecting the review dataset with other Olist datasets.

Missing values in these fields could prevent reliable relationships between tables. Therefore, we will first check whether either identifier contains missing values.

In [43]:
# Check missing values in identifier columns

review_id_columns = [
    "review_id",
    "order_id"
]

reviews_clean[review_id_columns].isna().sum()

review_id    0
order_id     0
dtype: int64

No missing values were found in either `review_id` or `order_id`.

This ensures that all review records contain the identifiers required to maintain relationships with the other Olist datasets.

No treatment was required for missing identifier values.

### 4.5.4.2 Check Duplicate Review IDs

The `review_id` column is expected to uniquely identify each review record.

Duplicate `review_id` values could indicate duplicated records or inconsistencies in the source data. We will therefore check whether any review IDs occur more than once.

In [44]:
# Check for duplicate review IDs

duplicate_review_ids = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
]

print(
    f"Rows with duplicated review_id: {len(duplicate_review_ids)}"
)

print(
    f"Number of unique duplicated review_id values: "
    f"{duplicate_review_ids['review_id'].nunique()}"
)

Rows with duplicated review_id: 1603
Number of unique duplicated review_id values: 789


### 4.5.4.3 Investigate Duplicate Review IDs

A total of 1,603 rows contain duplicated `review_id` values, corresponding to 789 unique review IDs.

A duplicated identifier does not automatically indicate that the records should be removed. The duplicated records may contain identical information or may represent different records associated with the same review ID.

To determine the appropriate treatment, we will compare the duplicated records across all columns.

In [45]:
# Display examples of duplicated review IDs

reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].sort_values(
    "review_id"
).head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


In [46]:
# Analyze the number of orders associated with each duplicated review_id

duplicate_review_analysis = (
    reviews_clean[
        reviews_clean["review_id"].duplicated(keep=False)
    ]
    .groupby("review_id")
    .agg(
        order_count=("order_id", "nunique"),
        row_count=("review_id", "size")
    )
)

duplicate_review_analysis.value_counts().sort_index()

order_count  row_count
2            2            764
3            3             25
Name: count, dtype: int64

In [47]:
print(
    "Duplicated review IDs with multiple orders:",
    (duplicate_review_analysis["order_count"] > 1).sum()
)

print(
    "Duplicated review IDs with only one order:",
    (duplicate_review_analysis["order_count"] == 1).sum()
)

Duplicated review IDs with multiple orders: 789
Duplicated review IDs with only one order: 0


### 4.5.4.3.1 Compare Duplicated Review Records

The investigation showed that all 789 duplicated `review_id` values are associated with multiple `order_id` values.

A total of 764 review IDs appear twice, while 25 appear three times. None of the duplicated review IDs are associated with the same order more than once.

This suggests that the duplicated `review_id` values may represent the same customer feedback associated with multiple orders.

To better understand this pattern, we will compare the remaining review attributes across duplicated records and determine whether the review content is consistent.

In [48]:
# Columns used to compare duplicated review records
comparison_columns = [
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

duplicate_reviews = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].copy()

# Count the number of unique values for each attribute
duplicate_reviews.groupby("review_id")[comparison_columns].nunique(
    dropna=False
).max().sort_values()

review_score               1
review_comment_title       1
review_comment_message     1
review_creation_date       1
review_answer_timestamp    1
dtype: int64

The investigation identified 789 duplicated `review_id` values involving 1,603 rows.

Among these duplicated review IDs:

- 764 appear twice.
- 25 appear three times.
- All duplicated review IDs are associated with different `order_id` values.
- None of the duplicated review IDs are associated with the same `order_id` more than once.

Further comparison showed that, within each duplicated `review_id`, the following attributes are identical:

- `review_score`
- `review_comment_title`
- `review_comment_message`
- `review_creation_date`
- `review_answer_timestamp`

The only attribute that differs is `order_id`.

This indicates that these records are not exact duplicate rows. Instead, the same review information is associated with multiple orders.

Because removing these records would eliminate valid `order_id` relationships, the duplicated `review_id` records will be retained.

The `review_id` column will therefore not be treated as a unique identifier in the cleaned dataset.

### 4.5.5 Check Duplicate Rows

Duplicated identifiers do not necessarily indicate duplicated records. Therefore, after investigating duplicated `review_id` values, we will perform a separate check for completely duplicated rows.

A completely duplicated row contains identical values across all columns. Such records would represent true duplicates and could potentially be removed without losing information.

The number of fully duplicated rows will be evaluated before deciding whether any treatment is necessary.

In [49]:
# Check for completely duplicated rows

duplicate_rows = reviews_clean[
    reviews_clean.duplicated(keep=False)
]

print(
    f"Number of duplicated rows: {len(duplicate_rows)}"
)

Number of duplicated rows: 0


In [50]:
print(
    f"Number of duplicate row groups: "
    f"{reviews_clean.duplicated().sum()}"
)

Number of duplicate row groups: 0


## 4.5.6 Check Invalid Values

After checking for duplicate rows, the next step is to verify whether the dataset contains values outside the expected range or format.

The `review_score` column should contain values between 1 and 5. Therefore, we will check for any values outside this range.

The text columns will also be checked for empty strings or values containing only whitespace.

In [51]:
# Check for invalid review scores
invalid_scores = reviews_clean[
    ~reviews_clean["review_score"].between(1, 5)
]

print(f"Number of invalid review scores: {len(invalid_scores)}")

Number of invalid review scores: 0


In [52]:
# Check for empty or whitespace-only values in text columns
text_columns = [
    "review_comment_title",
    "review_comment_message"
]

for col in text_columns:
    empty_values = (
        reviews_clean[col]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
    
    print(f"{col}: {empty_values} empty values")

review_comment_title: 87658 empty values
review_comment_message: 58274 empty values


In [53]:
# Check missing and empty values separately
for col in text_columns:
    missing_values = reviews_clean[col].isna().sum()
    empty_values = (
        reviews_clean[col]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
    
    actual_empty_values = empty_values - missing_values
    
    print(f"{col}:")
    print(f"  Missing values: {missing_values}")
    print(f"  Empty strings: {actual_empty_values}")

review_comment_title:
  Missing values: 87656
  Empty strings: 2
review_comment_message:
  Missing values: 58247
  Empty strings: 27


No invalid values were found in the `review_score` column. All review scores are within the expected range of 1 to 5.

The text columns contain a substantial number of missing values:

- `review_comment_title`: 87,656 missing values and 2 empty strings
- `review_comment_message`: 58,247 missing values and 27 empty strings

Missing review comments are expected in this dataset, as customers are not required to provide written feedback. Therefore, these missing values will be retained rather than removed.

Only a very small number of empty strings were identified, so they will be treated as missing values during the cleaning process.

In [54]:
# Convert empty or whitespace-only strings to NaN
for col in text_columns:
    reviews_clean[col] = reviews_clean[col].replace(r"^\s*$", np.nan, regex=True)

In [55]:
reviews_clean[text_columns].isna().sum()

review_comment_title      87658
review_comment_message    58274
dtype: int64

## 4.5.7 Check Date Consistency

The review dataset contains two date-related columns: `review_creation_date` and `review_answer_timestamp`.

To ensure chronological consistency, we will check whether any review response occurred before the corresponding review was created.

Records with an answer timestamp earlier than the review creation date would indicate a potential data quality issue and should be investigated.

In [56]:
# Check for inconsistent review dates
invalid_dates = reviews_clean[
    reviews_clean["review_answer_timestamp"] <
    reviews_clean["review_creation_date"]
]

print(f"Number of records with inconsistent dates: {len(invalid_dates)}")

Number of records with inconsistent dates: 0


No records were found with an answer timestamp earlier than the review creation date.

This confirms that the chronological relationship between `review_creation_date` and `review_answer_timestamp` is consistent, with no apparent date inconsistencies requiring treatment.

## 4.5.8 Final Validation

After completing the data cleaning and consistency checks, a final validation will be performed to confirm that the `reviews_clean` dataset is ready for exploratory data analysis.

The final validation will include:

- Dataset dimensions
- Data types
- Remaining missing values
- Duplicate rows
- Key variable validity

In [57]:
# Check final dataset dimensions
print(f"Rows: {reviews_clean.shape[0]}")
print(f"Columns: {reviews_clean.shape[1]}")

Rows: 99224
Columns: 7


In [58]:
# Check data types
reviews_clean.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [59]:
# Check remaining missing values
reviews_clean.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87658
review_comment_message     58274
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [60]:
# Check for fully duplicated rows
print(f"Number of duplicated rows: {reviews_clean.duplicated().sum()}")

Number of duplicated rows: 0


The final validation confirmed that the cleaned reviews dataset contains 99,224 rows and 7 columns.

The dataset has the following characteristics:

- `review_id`, `order_id`, and `review_score` contain no missing values.
- `review_creation_date` and `review_answer_timestamp` contain no missing values and are stored as datetime values.
- `review_comment_title` contains 87,658 missing values.
- `review_comment_message` contains 58,274 missing values.
- No fully duplicated rows were found.
- All `review_score` values are within the expected range of 1 to 5.
- No chronological inconsistencies were found between review creation and answer timestamps.

The missing values in the review comment fields were retained because written comments are optional and their absence does not indicate an invalid record.

The `reviews` dataset is now considered cleaned and validated and is ready for exploratory data analysis.

## 4.6 Products Data Cleaning

The products dataset contains information about the products available on the Olist marketplace, including product category, textual attributes, number of photos, weight, and dimensions.


### 4.6.1 Check Missing Values

The initial data inspection identified missing values in several product attributes.

A total of 610 products have missing values in `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`. In addition, two products have missing values in the physical attributes: `product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm`.

Before deciding how to treat these missing values, we will investigate whether the missing values occur in the same records.

In [61]:
# Check whether missing values occur in the same records
missing_product_attributes = products_clean[
    products_clean[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

missing_product_attributes

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [62]:
# Count missing values by row
missing_product_attributes.isna().sum(axis=1).value_counts().sort_index()

4    610
8      1
Name: count, dtype: int64

The missing value analysis identified 611 products affected by missing data.

Among these products:

- 610 products are missing `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`.
- 2 products are missing physical attributes (`product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm`).
- 1 product is missing all eight product attributes.

The missing values are therefore concentrated in a small number of products rather than being randomly distributed across the dataset.

Because the missing values represent unknown product information rather than confirmed zero values, they should not be replaced with zeros without further investigation.

### 4.6.2 Investigate Products with Missing Values

Before deciding how to treat the missing product attributes, we will determine whether the affected products appear in the order items dataset.

Products that were sold may contain relevant information for subsequent sales, revenue, or logistics analysis. Removing these products could therefore result in loss of useful information.

In [63]:
# Identify products affected by missing values
missing_product_ids = missing_product_attributes["product_id"]

# Check whether these products appear in order items
missing_products_in_orders = items_clean[
    items_clean["product_id"].isin(missing_product_ids)
]

print(f"Products with missing attributes: {len(missing_product_ids)}")
print(f"Order item records involving these products: {len(missing_products_in_orders)}")
print(
    f"Unique affected products found in orders: "
    f"{missing_products_in_orders['product_id'].nunique()}"
)

Products with missing attributes: 611
Order item records involving these products: 1604
Unique affected products found in orders: 611


All 611 products affected by missing values appear in the order items dataset, accounting for 1,604 order item records.

This indicates that the affected products were actively sold on the Olist marketplace.

Therefore, removing these products from the dataset would result in the loss of valid transactional information. The missing attributes will be retained and treated as unknown values rather than removing the affected products.

### 4.6.3 Assess the Impact of Missing Product Attributes

Since all products with missing attributes appear in order transactions, we will quantify their contribution to the marketplace sales.

This will help determine the potential impact of retaining these products with missing attributes in subsequent analyses.

In [64]:
# Calculate sales associated with products with missing attributes
affected_items = items_clean[
    items_clean["product_id"].isin(missing_product_ids)
]

affected_sales = affected_items["price"].sum()
total_sales = items_clean["price"].sum()

affected_sales_percentage = (affected_sales / total_sales) * 100

print(f"Sales from affected products: R$ {affected_sales:,.2f}")
print(f"Total sales from all products: R$ {total_sales:,.2f}")
print(f"Percentage of sales from affected products: {affected_sales_percentage:.2f}%")

Sales from affected products: R$ 181,469.28
Total sales from all products: R$ 13,591,643.70
Percentage of sales from affected products: 1.34%


The 611 products with missing attributes account for 1,604 order item records and R$181,469.28 in sales, representing approximately 1.34% of the total product sales value.

Although these products represent a relatively small proportion of total sales, they are associated with valid transactions and should therefore be retained.

The missing attributes will be treated as unknown values rather than removing the affected products or imputing values that cannot be reliably determined from the available data.

In [65]:
# Identify products with missing physical attributes
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_physical_attributes = products_clean[
    products_clean[physical_columns].isna().any(axis=1)
]

print(
    f"Products with missing physical attributes: "
    f"{len(missing_physical_attributes)}"
)

print(
    f"Products also missing category-related attributes: "
    f"{missing_physical_attributes['product_id'].isin(missing_product_ids).sum()}"
)

Products with missing physical attributes: 2
Products also missing category-related attributes: 2


Two products were found with missing physical attributes. Both products are already included among the 611 products with missing category-related attributes.

Therefore, the missing values are concentrated within the same group of affected products, and no additional products need to be treated.

Since these products are associated with valid transactions, they will be retained in the dataset. Missing attributes will remain as unknown values rather than being removed or replaced with estimated values.

### 4.6.5 Check Invalid Values

After reviewing the missing values, the next step is to identify values that fall outside the expected ranges of the product attributes.

The following attributes will be checked:

- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`
- `product_weight_g`
- `product_length_cm`
- `product_height_cm`
- `product_width_cm`

Values equal to or below zero will be investigated to determine whether they represent valid observations or potential data quality issues.

In [66]:
# Check for non-positive values
numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_columns:
    invalid_count = (products_clean[col] <= 0).sum()
    print(f"{col}: {invalid_count} non-positive values")

product_name_lenght: 0 non-positive values
product_description_lenght: 0 non-positive values
product_photos_qty: 0 non-positive values
product_weight_g: 4 non-positive values
product_length_cm: 0 non-positive values
product_height_cm: 0 non-positive values
product_width_cm: 0 non-positive values


No non-positive values were found in `product_name_lenght`, `product_description_lenght`, `product_photos_qty`, or any of the product dimension columns.

Four records were identified with non-positive values in `product_weight_g`. These records require further investigation before deciding whether the values should be treated as invalid.

In [67]:
# Inspect products with non-positive weight
invalid_weight_products = products_clean[
    products_clean["product_weight_g"] <= 0
]

invalid_weight_products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [68]:
# Check whether products with zero weight appear in orders
invalid_weight_product_ids = invalid_weight_products["product_id"]

invalid_weight_items = items_clean[
    items_clean["product_id"].isin(invalid_weight_product_ids)
]

print(f"Products with zero weight: {len(invalid_weight_product_ids)}")
print(f"Order item records involving these products: {len(invalid_weight_items)}")
print(
    f"Unique products found in orders: "
    f"{invalid_weight_items['product_id'].nunique()}"
)

Products with zero weight: 4
Order item records involving these products: 8
Unique products found in orders: 4


All four products with a recorded weight of 0 grams appear in the order items dataset, accounting for 8 order item records.

Since these products were sold and have valid product dimensions, the zero-weight values are considered data quality issues rather than valid measurements.

The products will be retained, while the zero values in `product_weight_g` will be treated as missing values.

In [69]:
# Calculate the sales associated with zero-weight products
zero_weight_sales = invalid_weight_items["price"].sum()
total_sales = items_clean["price"].sum()

zero_weight_sales_percentage = (
    zero_weight_sales / total_sales
) * 100

print(f"Sales from zero-weight products: R$ {zero_weight_sales:,.2f}")
print(f"Total sales from all products: R$ {total_sales:,.2f}")
print(
    f"Percentage of sales from zero-weight products: "
    f"{zero_weight_sales_percentage:.2f}%"
)

Sales from zero-weight products: R$ 949.50
Total sales from all products: R$ 13,591,643.70
Percentage of sales from zero-weight products: 0.01%


In [70]:
# Replace zero weights with NaN
products_clean.loc[
    products_clean["product_weight_g"] <= 0,
    "product_weight_g"
] = np.nan

In [71]:
# Verify that no non-positive weight values remain
print(
    "Non-positive product weights:",
    (products_clean["product_weight_g"] <= 0).sum()
)

print(
    "Missing product weights:",
    products_clean["product_weight_g"].isna().sum()
)

Non-positive product weights: 0
Missing product weights: 6


## 4.6.6 Final Validation

After treating the identified missing and invalid values, a final validation will be performed to confirm that the `products_clean` dataset is ready for further analysis.

The validation will check for remaining invalid values, missing values, duplicate rows, and the final dataset structure.

In [72]:
# Check for remaining non-positive values
for col in numeric_columns:
    invalid_count = (products_clean[col] <= 0).sum()
    print(f"{col}: {invalid_count} non-positive values")

product_name_lenght: 0 non-positive values
product_description_lenght: 0 non-positive values
product_photos_qty: 0 non-positive values
product_weight_g: 0 non-positive values
product_length_cm: 0 non-positive values
product_height_cm: 0 non-positive values
product_width_cm: 0 non-positive values


In [73]:
# Check remaining missing values
products_clean.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                6
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [74]:
# Check for fully duplicated rows
print(
    f"Number of fully duplicated rows: "
    f"{products_clean.duplicated().sum()}"
)

Number of fully duplicated rows: 0


In [75]:
# Check final dataset dimensions
print(f"Rows: {products_clean.shape[0]}")
print(f"Columns: {products_clean.shape[1]}")

Rows: 32951
Columns: 9


In [76]:
# Check final data types
products_clean.dtypes

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

The final validation confirmed that the `products_clean` dataset contains 32,951 rows and 9 columns.

The cleaning process resulted in the following:

- No duplicated product IDs were found.
- No fully duplicated rows were found.
- No non-positive values remain in the numeric product attributes.
- 610 products retain missing values in `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`.
- 6 products have missing `product_weight_g` values after four invalid zero-weight values were converted to `NaN`.
- 2 products have missing values in each physical dimension column.
- `product_id` contains no missing values.
- All date-independent product attributes have appropriate data types.

The products with missing attributes were retained because they are associated with valid order transactions. Missing values were not imputed where reliable replacement values could not be determined from the available data.

The `products_clean` dataset is now considered cleaned and validated and is ready for further analysis.

## 4.7 Sellers Data Cleaning

The sellers dataset contains no missing values across any of its columns.

Since all seller records contain complete information, no missing value treatment is required at this stage.

### 4.7.1 Validate Seller State Values

The `seller_state` column contains the state abbreviation associated with each seller.

Since Brazilian states use standardized two-letter abbreviations, the values will be checked against the expected Brazilian state and Federal District codes.

In [77]:
# Expected Brazilian state and Federal District abbreviations
valid_states = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
}

invalid_states = sellers_clean[
    ~sellers_clean["seller_state"].isin(valid_states)
]

print(f"Number of invalid state values: {len(invalid_states)}")

Number of invalid state values: 0


No invalid seller state values were identified. All values in `seller_state` correspond to valid Brazilian state or Federal District abbreviations.

### 4.7.2 Final Validation

After completing the data quality checks, a final validation will be performed to confirm that the `sellers_clean` dataset is complete, unique, and ready for further analysis.

In [78]:
# Check final dataset dimensions
print(f"Rows: {sellers_clean.shape[0]}")
print(f"Columns: {sellers_clean.shape[1]}")

Rows: 3095
Columns: 4


In [79]:
# Check remaining missing values
sellers_clean.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [80]:
# Check for fully duplicated rows
print(
    f"Number of fully duplicated rows: "
    f"{sellers_clean.duplicated().sum()}"
)

Number of fully duplicated rows: 0


In [81]:
# Check seller ID uniqueness
print(
    f"Unique seller IDs: "
    f"{sellers_clean['seller_id'].nunique()}"
)

Unique seller IDs: 3095


In [82]:
# Check data types
sellers_clean.dtypes

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

The final validation confirmed that the `sellers_clean` dataset contains 3,095 rows and 4 columns.

The cleaning process resulted in the following:

- No missing values were found.
- No fully duplicated rows were found.
- All `seller_id` values are unique.
- No invalid Brazilian state abbreviations were identified.
- `seller_zip_code_prefix` is stored as an integer and represents a ZIP code prefix.
- `seller_city` and `seller_state` are stored as string values.

No records required removal or modification during the cleaning process.

The `sellers_clean` dataset is considered clean and validated and is ready for further analysis.

## 4.8 Category Data Cleaning

The category dataset contains the relationship between the original Portuguese product category names and their English translations.

The cleaning process will focus on checking missing values, duplicate mappings, and the consistency of the category translation pairs.

### 4.8.1 Check Translation Consistency

Each Portuguese category should map to a single English category name.

Multiple English translations for the same Portuguese category could create ambiguity when joining the category dataset with the products dataset.

In [83]:
# Check whether each Portuguese category has a unique English translation
category_mapping_counts = (
    category_clean
    .groupby("product_category_name")
    ["product_category_name_english"]
    .nunique()
)

inconsistent_mappings = category_mapping_counts[
    category_mapping_counts > 1
]

print(
    f"Number of categories with multiple English translations: "
    f"{len(inconsistent_mappings)}"
)

Number of categories with multiple English translations: 0


### 4.8.2 Final Validation

The final validation will confirm that the category mapping is complete, unique, and consistent before the dataset is used to translate product categories.

In [84]:
print(f"Rows: {category_clean.shape[0]}")
print(f"Columns: {category_clean.shape[1]}")
print(f"Missing values: {category_clean.isna().sum().sum()}")
print(f"Duplicated rows: {category_clean.duplicated().sum()}")

Rows: 71
Columns: 2
Missing values: 0
Duplicated rows: 0


In [85]:
# Check whether each Portuguese category maps to exactly one English category
print(
    f"Categories with inconsistent mappings: "
    f"{len(inconsistent_mappings)}"
)

Categories with inconsistent mappings: 0


The final validation confirmed that the category dataset contains 71 rows and 2 columns.

The cleaning process resulted in the following:

- No missing values were found.
- No fully duplicated rows were found.
- Each Portuguese category maps to a single English category.
- No inconsistent category translations were identified.

No records required removal or modification during the cleaning process.

The `category_clean` dataset is considered clean and validated and is ready to be used for translating product category names in subsequent analysis.

## 4.9 Geolocation Dataset

The geolocation dataset contains ZIP code prefixes and their corresponding geographic coordinates and location information.

The cleaning process will focus on identifying missing values, duplicate records, and potential inconsistencies in ZIP code prefixes, latitude, longitude, city, and state attributes.

### 4.9.1 Check Duplicate Records

The geolocation dataset contains multiple records associated with the same ZIP code prefix. Therefore, duplicated identifiers do not necessarily represent duplicated records.

However, completely duplicated rows represent identical geographic observations and may be safely removed if they do not provide additional information.

The number of fully duplicated rows will be evaluated before deciding how to treat them.

In [86]:
# Count fully duplicated rows
duplicate_rows = geolocation_clean[
    geolocation_clean.duplicated(keep=False)
]

print(f"Number of rows involved in duplicate groups: {len(duplicate_rows)}")
print(
    f"Number of duplicate rows excluding the first occurrence: "
    f"{geolocation_clean.duplicated().sum()}"
)

Number of rows involved in duplicate groups: 390005
Number of duplicate rows excluding the first occurrence: 261831


In [87]:
# Count unique duplicate groups
duplicate_group_count = (
    geolocation_clean[
        geolocation_clean.duplicated(keep=False)
    ]
    .drop_duplicates()
    .shape[0]
)

print(f"Number of unique duplicate row groups: {duplicate_group_count}")

Number of unique duplicate row groups: 128174


The geolocation dataset contains 261,831 rows that are exact duplicates of previously occurring records.

A total of 390,005 rows belong to duplicate groups, meaning that these records are repeated at least once within the dataset.

Since the duplicated records contain identical values across all columns, they do not provide additional geographic information. Therefore, exact duplicate rows can be removed without losing unique observations.

In [88]:
# Compare the dataset before and after removing exact duplicates
geolocation_unique = geolocation_clean.drop_duplicates()

print(f"Original rows: {len(geolocation_clean)}")
print(f"Unique rows: {len(geolocation_unique)}")
print(
    f"Duplicate rows removed: "
    f"{len(geolocation_clean) - len(geolocation_unique)}"
)

Original rows: 1000163
Unique rows: 738332
Duplicate rows removed: 261831


The dataset initially contained 1,000,163 rows. After removing 261,831 exact duplicate rows, 738,332 unique geolocation records remained.

Because the duplicated rows contained identical values across all columns, their removal did not result in the loss of unique geographic information.

### 4.9.2 Validate Geographic Coordinates

After removing exact duplicate rows, the geographic coordinates were validated to ensure that all latitude and longitude values fall within their valid geographic ranges.

No invalid latitude or longitude values were identified.

In [89]:
# Check for invalid latitude values
invalid_latitude = geolocation_unique[
    (geolocation_unique["geolocation_lat"] < -90) |
    (geolocation_unique["geolocation_lat"] > 90)
]

print(f"Number of invalid latitude values: {len(invalid_latitude)}")

Number of invalid latitude values: 0


In [90]:
# Check for invalid longitude values
invalid_longitude = geolocation_unique[
    (geolocation_unique["geolocation_lng"] < -180) |
    (geolocation_unique["geolocation_lng"] > 180)
]

print(f"Number of invalid longitude values: {len(invalid_longitude)}")

Number of invalid longitude values: 0


### 4.9.3 Validate ZIP Code and State Consistency

A ZIP code prefix should generally correspond to a specific geographic region.

Multiple records may exist for the same ZIP code prefix because a prefix can cover multiple geographic points. However, a single ZIP code prefix being associated with multiple states may indicate a potential data quality issue.

Therefore, the relationship between `geolocation_zip_code_prefix` and `geolocation_state` will be investigated.

In [91]:
# Count the number of states associated with each ZIP code prefix
zip_state_counts = (
    geolocation_unique
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

inconsistent_zip_states = zip_state_counts[
    zip_state_counts > 1
]

print(
    f"Number of ZIP code prefixes associated with multiple states: "
    f"{len(inconsistent_zip_states)}"
)

Number of ZIP code prefixes associated with multiple states: 8


In [92]:
# Inspect ZIP code prefixes associated with multiple states
inconsistent_records = geolocation_unique[
    geolocation_unique["geolocation_zip_code_prefix"].isin(
        inconsistent_zip_states.index
    )
].sort_values(
    [
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ]
)

inconsistent_records[
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "geolocation_lat",
        "geolocation_lng"
    ]
]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state,geolocation_lat,geolocation_lng
22261,2116,são paulo,RN,-23.515978,-46.582170
21728,2116,sao paulo,SP,-23.522700,-46.587546
21759,2116,sao paulo,SP,-23.518459,-46.584128
22060,2116,sao paulo,SP,-23.519065,-46.584610
22193,2116,são paulo,SP,-23.519739,-46.585151
...,...,...,...,...,...
847857,80630,curitiba,PR,-25.471820,-49.281597
847872,80630,curitiba,PR,-25.469776,-49.273230
847875,80630,curitiba,PR,-25.466519,-49.268213
847897,80630,curitiba,PR,-25.473179,-49.282526


In [93]:
# Count records for each ZIP prefix and state combination
zip_state_summary = (
    inconsistent_records
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    )
    .size()
    .reset_index(name="record_count")
)

zip_state_summary

,geolocation_zip_code_prefix,geolocation_state,record_count
0,2116,RN,1
1,2116,SP,10
2,4011,AC,1
3,4011,SP,69
4,21550,AC,1
5,21550,RJ,144
6,23056,AC,1
7,23056,RJ,30
8,72915,DF,1
9,72915,GO,9


### 4.9.4 Investigate Inconsistent ZIP Code Prefixes

Eight ZIP code prefixes were found to be associated with multiple states.

For each affected ZIP code prefix, one state occurs only once, while another state accounts for the majority of records. These isolated records will be investigated using their city and geographic coordinates before determining the appropriate treatment.

In [94]:
# Identify state combinations that occur only once
minority_records = zip_state_summary[
    zip_state_summary["record_count"] == 1
]

minority_records

,geolocation_zip_code_prefix,geolocation_state,record_count
0,2116,RN,1
2,4011,AC,1
4,21550,AC,1
6,23056,AC,1
8,72915,DF,1
11,78557,RO,1
13,79750,RS,1
15,80630,SC,1


In [95]:
# Retrieve the complete records for the minority state combinations
minority_details = inconsistent_records.merge(
    minority_records[
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    ],
    on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="inner"
)

minority_details[
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "geolocation_lat",
        "geolocation_lng"
    ]
]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state,geolocation_lat,geolocation_lng
0,2116,são paulo,RN,-23.515978,-46.582170
1,4011,sao paulo,AC,-23.578707,-46.645779
2,21550,rio de janeiro,AC,-22.857861,-43.352613
3,23056,rio de janeiro,AC,-22.919164,-43.611097
4,72915,taguatinga,DF,-12.408760,-46.428287
5,78557,nova brasilandia d'oeste,RO,-11.875142,-55.503020
6,79750,nova andradina,RS,-22.242062,-53.343159
7,80630,balneario de picarras,SC,-26.757371,-48.675738


### 4.9.5 Validate City and State Consistency

The previous analysis identified eight ZIP code prefixes associated with multiple states. However, the most frequent state for a ZIP code prefix cannot automatically be assumed to be correct.

To avoid introducing incorrect corrections, the relationship between city and state will also be examined. This will help determine whether the isolated records are consistent with the geographic information contained in the dataset.

In [96]:
# Count the number of records for each city and state combination
city_state_summary = (
    inconsistent_records
    .groupby(
        [
            "geolocation_city",
            "geolocation_state"
        ]
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["geolocation_city", "record_count"],
        ascending=[True, False]
    )
)

city_state_summary

,geolocation_city,geolocation_state,record_count
0,aguas lindas de goias,GO,7
1,balneario de picarras,SC,1
2,curitiba,PR,76
3,nova andradina,MS,151
4,nova andradina,RS,1
5,nova brasilandia d'oeste,RO,1
7,rio de janeiro,RJ,174
6,rio de janeiro,AC,2
9,sao paulo,SP,58
8,sao paulo,AC,1


In [97]:
# Count the number of states associated with each city
city_state_counts = (
    inconsistent_records
    .groupby("geolocation_city")["geolocation_state"]
    .nunique()
)

city_state_counts

geolocation_city
aguas lindas de goias       1
balneario de picarras       1
curitiba                    1
nova andradina              2
nova brasilandia d'oeste    1
rio de janeiro              2
sao paulo                   2
sinop                       1
são paulo                   2
taguatinga                  1
águas lindas de goiás       1
Name: geolocation_state, dtype: int64

### 4.9.6 Remove Inconsistent Geographic Records

The investigation identified eight records with inconsistent ZIP code and state combinations.

The inconsistencies were confirmed by comparing the reported city, state, and geographic coordinates. In each case, the geographic coordinates were inconsistent with the reported state.

These eight records will be removed because their geographic information cannot be considered reliable.

In [98]:
# Create a mask identifying the eight inconsistent records
inconsistent_mask = (
    geolocation_unique[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state",
            "geolocation_lat",
            "geolocation_lng"
        ]
    ]
    .astype(str)
    .agg("|".join, axis=1)
    .isin(
        minority_details[
            [
                "geolocation_zip_code_prefix",
                "geolocation_city",
                "geolocation_state",
                "geolocation_lat",
                "geolocation_lng"
            ]
        ]
        .astype(str)
        .agg("|".join, axis=1)
    )
)

print(f"Records identified for removal: {inconsistent_mask.sum()}")

Records identified for removal: 8


In [99]:
# Remove the eight inconsistent geographic records
geolocation_clean = (
    geolocation_unique[
        ~inconsistent_mask
    ]
    .reset_index(drop=True)
)

print(f"Rows after removing inconsistent records: {len(geolocation_clean)}")

Rows after removing inconsistent records: 738324


### 4.9.7 Final Validation

After removing the inconsistent geographic records, the cleaned dataset was validated again to ensure that the expected number of records remained and that no invalid geographic coordinates were introduced.

The final cleaned geolocation dataset contains 738,324 records.

In [100]:
# Final validation of the cleaned geolocation dataset

print(f"Rows: {len(geolocation_clean)}")
print(f"Columns: {geolocation_clean.shape[1]}")

print(
    f"Missing values: "
    f"{geolocation_clean.isnull().sum().sum()}"
)

print(
    f"Fully duplicated rows: "
    f"{geolocation_clean.duplicated().sum()}"
)

print(
    f"Invalid latitude values: "
    f"{((geolocation_clean["geolocation_lat"] < -90) | (geolocation_clean["geolocation_lat"] > 90)).sum()}"
)

print(
    f"Invalid longitude values: "
    f"{((geolocation_clean["geolocation_lng"] < -180) | (geolocation_clean["geolocation_lng"] > 180)).sum()}"
)

Rows: 738324
Columns: 5
Missing values: 0
Fully duplicated rows: 0
Invalid latitude values: 0
Invalid longitude values: 0


## 5. Data Cleaning Summary

The data cleaning process was performed to improve data quality while preserving valid business information. Each dataset was evaluated individually, and treatment decisions were based on the nature and potential impact of the identified issues.

### Main Cleaning Decisions

- **Customers:** No records required removal. The dataset contained no missing values or fully duplicated rows.

- **Orders:** Date columns were converted to datetime format. Missing dates were investigated in relation to order status and were retained when they represented valid business situations.

- **Order Items:** No missing values or duplicate records were identified. The dataset was retained without row removal.

- **Payments:** Invalid payment records were investigated based on payment type, payment value, and order status. Records identified as invalid were removed.

- **Reviews:** Invalid review scores were not found. Missing review comments were retained because written comments are optional fields. Date consistency was also validated.

- **Products:** Missing product attributes were investigated in relation to order activity and sales. Records were retained to avoid losing valid product and sales information. Non-positive product weights were investigated separately.

- **Sellers:** No missing values, invalid state values, or fully duplicated rows were identified. The dataset was retained without row removal.

- **Product Categories:** No missing values, duplicate rows, or inconsistent category-to-translation mappings were identified.

- **Geolocation:** Exact duplicate records were removed. Geographic coordinates were validated, and eight records with confirmed inconsistencies between ZIP code, city, state, and geographic coordinates were removed.

### Data Preservation Principle

Throughout the cleaning process, records were not removed solely because they contained missing or unusual values. Each issue was investigated in the context of the dataset and its relationship with other tables before a treatment decision was made.

This approach was used to minimize unnecessary data loss while ensuring that the final datasets were suitable for exploratory analysis and business analysis.

### Final Dataset Overview

| Dataset | Original Rows | Final Rows | Main Treatment |
|---|---:|---:|---|
| Customers | 99,441 | 99,441 | No row removal |
| Orders | 99,441 | 99,441 | Date conversion and validation |
| Order Items | 112,650 | 112,650 | No row removal |
| Payments | 103,886 | 103,883 | Invalid records removed |
| Reviews | 99,224 | 99,224 | Missing comments retained |
| Products | 32,951 | 32,951 | Missing attributes investigated |
| Sellers | 3,095 | 3,095 | No row removal |
| Product Categories | 71 | 71 | No row removal |
| Geolocation | 1,000,163 | 738,324 | Duplicate and inconsistent records removed |

## 6. Export Cleaned Datasets

After completing the data cleaning process, the cleaned datasets are exported as CSV files.

These files will be used as the input data for the Exploratory Data Analysis performed in the next notebook.

Keeping the cleaned datasets separate from the original files ensures that the original data remains unchanged and makes the analysis workflow reproducible.

In [101]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

customers_clean.to_csv(PROCESSED_DIR / "customers_clean.csv", index=False)
orders_clean.to_csv(PROCESSED_DIR / "orders_clean.csv", index=False)
items_clean.to_csv(PROCESSED_DIR / "items_clean.csv", index=False)
payments_clean.to_csv(PROCESSED_DIR / "payments_clean.csv", index=False)
reviews_clean.to_csv(PROCESSED_DIR / "reviews_clean.csv", index=False)
products_clean.to_csv(PROCESSED_DIR / "products_clean.csv", index=False)
sellers_clean.to_csv(PROCESSED_DIR / "sellers_clean.csv", index=False)
category_clean.to_csv(PROCESSED_DIR / "category_clean.csv", index=False)
geolocation_clean.to_csv(PROCESSED_DIR / "geolocation_clean.csv", index=False)